# Upload Dataset to HuggingFace Hub

This notebook loads the `eng_dataset.csv` file (disease classification dataset with `label` and `text` columns), validates it, and uploads it to HuggingFace Hub using the `datasets` library.

## 1. Install and Import Required Libraries

In [1]:
%pip install datasets huggingface_hub pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict
from huggingface_hub import login
from getpass import getpass

/home/notlath/Documents/Thesis/aill-be-sick/backend/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure HuggingFace Authentication

Log in to HuggingFace Hub using your API token. The cell below uses `getpass` so your token is never displayed in the notebook output. Paste your token when prompted.

In [3]:
# Securely input your HuggingFace token (will not be displayed)
hf_token = getpass("Enter your HuggingFace API token: ")
login(token=hf_token)

## 3. Load the Local Dataset

Load `eng_dataset.csv` which contains disease classification data with two columns:
- **label**: The disease name (Dengue, Influenza, Measles, Pneumonia, Typhoid, Diarrhea)
- **text**: Patient symptom descriptions

In [4]:
df = pd.read_csv("fil_dataset.csv")
print(f"Loaded dataset with {len(df)} rows")
df.head(10)

Loaded dataset with 6000 rows


,label,text
0,Typhoid,"Araw-araw lumalala yung init ng lagnat ko, pan..."
1,Measles,May sinat po na 38.6°C ng ilang araw. Hindi na...
2,Measles,Ilang araw na po akong nilalagnat. May kasama ...
3,Diarrhea,Mga apat na beses na akong nagtatae ngayong ar...
4,Pneumonia,"Sobra na itong lagnat at ginaw ko, kahit wala ..."
5,Dengue,Sakit ng kasu-kasuan. Sobrang sakit ng katawan...
6,Pneumonia,"Ilang araw na masama pakiramdam ko, may lagnat..."
7,Pneumonia,"Dahil hirap huminga at mabilis hingalin, pumun..."
8,Influenza,Nanonood lang ako ng TV nung biglang sumakit y...
9,Pneumonia,Talagang kinakapos ako ng hininga nang malala ...


## 4. Explore and Validate the Dataset

Check shape, column types, class distribution, and missing values.

In [5]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df['label'].value_counts()}")

Shape: (6000, 2)

Column types:
label    str
text     str
dtype: object

Missing values:
label    0
text     0
dtype: int64

Class distribution:
label
Typhoid      1000
Measles      1000
Diarrhea     1000
Pneumonia    1000
Dengue       1000
Influenza    1000
Name: count, dtype: int64


In [6]:
# Preview a sample from each class
df.groupby("label").apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)

,text
0,"Nagsimula sa sinat lang nung isang araw, pero ..."
1,Pitong beses na akong nag-loose bowel mula kag...
2,Biglaan kong naramdaman itong init ng katawan ...
3,May lagnat po na 39.5°C sa loob ng tatlong ara...
4,"Bale nag-start sa pagtatae, tapos ngayon inuub..."
5,"Matagal na itong lagnat ko, lampas sampung ara..."


## 5. Create a HuggingFace Dataset Object

Convert the DataFrame to a HuggingFace `Dataset` and optionally split into train/test sets.

In [7]:

from datasets import ClassLabel, Features, Value

# Build label mapping
label_names = sorted(df["label"].unique().tolist())
label2id = {name: i for i, name in enumerate(label_names)}

# Encode labels as integers first (avoids ChunkedArray cast bug in datasets/PyArrow)
df_encoded = df.copy()
df_encoded["label"] = df_encoded["label"].map(label2id)

features = Features({
    "label": ClassLabel(names=label_names),
    "text": Value("string"),
})

# Convert pandas DataFrame to HuggingFace Dataset with typed features
dataset = Dataset.from_pandas(df_encoded, features=features)
print(dataset)

# Split into 70% train, 30% test (stratified)
dataset_dict = dataset.train_test_split(test_size=0.3, seed=42, stratify_by_column="label")
print(f"\nTrain: {dataset_dict['train'].num_rows} rows")
print(f"Test:  {dataset_dict['test'].num_rows} rows")


Dataset({
    features: ['label', 'text'],
    num_rows: 6000
})

Train: 4200 rows
Test:  1800 rows


## 6. Push Dataset to HuggingFace Hub

Upload the dataset to your HuggingFace repository. Replace `"your-username/eng_dataset"` with your actual HuggingFace username and desired repo name.

In [8]:
# Set your HuggingFace repo name (format: "username/dataset-name")
REPO_NAME = "notlath/fil_dataset"

# Push the train/test split dataset to HuggingFace Hub
dataset_dict.push_to_hub(REPO_NAME, token=hf_token)

print(f"Dataset successfully uploaded to https://huggingface.co/datasets/{REPO_NAME}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 21.79ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  95%|█████████▌|  525kB /  550kB,  437kB/s  
















Processing Files (1 / 1)                : 100%|██████████|  550kB /  550kB,  120kB/s  


Processing Files (1 / 1)                : 100%|██████████|  550kB /  550kB,  110kB/s  
New Data Upload                         : 100%|██████████|  550kB /  550kB,  110kB/s  
  /tmp/tmpxfvskk4e.parquet              : 100%|██████████|  550kB /  550kB            
Uploading the dataset shards: 100%|██████████| 1/1 [00:06<00:00,  6.36s/ shards]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 33.00b

Dataset successfully uploaded to https://huggingface.co/datasets/notlath/fil_dataset
